# Understanding Claude Code Token Accounting

### Why your token-counting tool disagrees with `/usage` — and what's *actually* being measured

I built [claude-watch](https://github.com/odedha-dr/claude-watch) — a dashboard that watches Claude Code's session logs and shows token usage live. It worked great, until I compared its numbers to Claude Code's built-in `/usage` command and they disagreed wildly.

Some fields were 3× too high. Others were 13× too *low*. The total cost happened to be within 2% — but the per-field breakdown was a mess.

This notebook walks through the investigation: where Claude Code's `/usage` actually gets its numbers, what the JSONL files contain, what the gap means, and how to bring them into alignment.

Run it on **your own** Claude Code session — set `JSONL_PATH` below and step through.

---

**TL;DR for the impatient:**

| Metric | JSONL-derived | `/usage` daemon | Why they differ |
|---|---|---|---|
| Total Cost | ✓ matches within 2% | matches | dominated by cache_read |
| Cache Read | ✓ matches within 1% | matches | summed cleanly per turn |
| Output tokens | ~25% over | counts narrower | thinking-token redaction in JSONL |
| Cache Write | ~47% over | ephemeral-TTL accounting | 1h vs 5m bucket normalization |
| Input tokens | 92% **under** | includes auxiliary calls | title-gen / agent-name API calls aren't persisted to JSONL |

Plus: assistant messages are split into multiple JSONL entries (one per content block), each carrying the same `usage` object. If your tool doesn't dedup, it 2-3× counts everything.


## Setup

Stdlib only — no external dependencies. Set `JSONL_PATH` to your own session file (or use the auto-discover helper to grab the latest).

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

HOME = Path.home()
PROJECTS = HOME / '.claude' / 'projects'

def find_recent_jsonls(n=5):
    """Auto-discover the N most recently modified Claude Code session JSONLs."""
    files = []
    if PROJECTS.exists():
        for proj in PROJECTS.iterdir():
            if proj.is_dir():
                for f in proj.glob('*.jsonl'):
                    files.append((f.stat().st_mtime, f))
    files.sort(reverse=True)
    return [f for _, f in files[:n]]

for f in find_recent_jsonls(5):
    size_kb = f.stat().st_size // 1024
    mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
    print(f'{mtime}  {size_kb:>6} KB  {f}')

In [ ]:
# Pick a session to analyze. Either paste an explicit path or use the most recent.
JSONL_PATH = find_recent_jsonls(1)[0]
# JSONL_PATH = Path('/path/to/your/session.jsonl')
print(f'Analyzing: {JSONL_PATH}')

with open(JSONL_PATH) as f:
    LINES = [line for line in f if line.strip()]

ENTRIES = []
for line in LINES:
    try:
        ENTRIES.append(json.loads(line))
    except json.JSONDecodeError:
        pass
print(f'{len(ENTRIES):,} JSONL entries loaded')

## Step 1 — What's actually in the file?

Before we count tokens, let's see what types of entries Claude Code writes. The `type` field is the top-level discriminator.

In [ ]:
type_counts = Counter()
subtype_counts = Counter()
for e in ENTRIES:
    type_counts[e.get('type', '?')] += 1
    if e.get('subtype'):
        subtype_counts[f"{e.get('type')}/{e.get('subtype')}"] += 1

print('Entry types:')
for t, n in type_counts.most_common():
    print(f'  {t:30} {n:>5}')
if subtype_counts:
    print('\nSubtypes:')
    for st, n in subtype_counts.most_common():
        print(f'  {st:30} {n:>5}')

The interesting ones for token accounting:

- **`assistant`** — the model's responses. These carry `message.usage` with input/output/cache tokens.
- **`user`** — your prompts (plus tool results). No `usage` field; their content becomes input for the *next* assistant turn.
- **`ai-title` / `agent-name`** — Claude Code makes separate API calls to generate session titles and agent names. The *results* land here but the per-call `usage` is **not persisted**. This is the source of the biggest discrepancy we'll uncover later.
- **`system` / `compact_boundary`** — context compactions.
- **`last-prompt` / `agent-setting` / `permission-mode` / `file-history-snapshot`** — metadata bookkeeping; no token counts.

Only `assistant` entries have per-call usage data we can sum.

## Step 2 — The naive sum (and why it's wrong)

First-pass: walk every `assistant` entry and sum its `usage` fields.

In [ ]:
def empty_usage():
    return {'input': 0, 'output': 0, 'cache_create': 0, 'cache_read': 0}

naive = empty_usage()
n_assistant = 0
for e in ENTRIES:
    msg = e.get('message', {}) or {}
    if msg.get('role') != 'assistant':
        continue
    u = msg.get('usage', {}) or {}
    naive['input']        += u.get('input_tokens', 0) or 0
    naive['output']       += u.get('output_tokens', 0) or 0
    naive['cache_create'] += u.get('cache_creation_input_tokens', 0) or 0
    naive['cache_read']   += u.get('cache_read_input_tokens', 0) or 0
    n_assistant += 1

print(f'Naive sum across {n_assistant} assistant entries:')
for k, v in naive.items():
    print(f'  {k:14} {v:>15,}')

Now look at how many of those `assistant` entries share the same `message.id`. In a clean log, every assistant message gets logged once. Let's check.

In [ ]:
msg_id_counts = Counter()
for e in ENTRIES:
    msg = e.get('message', {}) or {}
    if msg.get('role') == 'assistant':
        mid = msg.get('id')
        if mid:
            msg_id_counts[mid] += 1

fragment_dist = Counter(msg_id_counts.values())
print(f'Unique message IDs: {len(msg_id_counts)}')
print(f'Total assistant entries: {sum(msg_id_counts.values())}')
print(f'\nFragments per message id:')
for n_frag, n_msgs in sorted(fragment_dist.items()):
    bar = '\u2588' * min(40, n_msgs)
    print(f'  {n_frag} fragment(s): {n_msgs:>3} messages  {bar}')

**This is the first finding.** Many `message.id` values appear *more than once*. Claude Code splits one assistant message into multiple JSONL entries — one per content block (`thinking` / `text` / `tool_use`) — and **every fragment carries the same `usage` object**.

Inspect one to confirm:

In [ ]:
# Find the first message with multiple fragments
target_mid = next(mid for mid, n in msg_id_counts.items() if n > 1)
print(f'Inspecting fragments of {target_mid}:\n')
for i, e in enumerate(ENTRIES):
    msg = e.get('message', {}) or {}
    if msg.get('role') == 'assistant' and msg.get('id') == target_mid:
        content = msg.get('content') or []
        block_summary = []
        for c in content:
            t = c.get('type')
            if t == 'text': block_summary.append(f'text({len(c.get("text", ""))} chars)')
            elif t == 'thinking': block_summary.append(f'thinking({len(c.get("thinking", ""))} chars)')
            elif t == 'tool_use': block_summary.append(f'tool_use:{c.get("name")}')
            else: block_summary.append(t)
        u = msg.get('usage', {}) or {}
        print(f'  line {i+1}: blocks={block_summary}')
        print(f'           usage={dict(u)}')

All fragments report the same `usage`. A naive sum counts the same tokens 2-3 times.

## Step 3 — Dedup by `message.id`

In [ ]:
deduped = empty_usage()
seen = set()
for e in ENTRIES:
    msg = e.get('message', {}) or {}
    if msg.get('role') != 'assistant':
        continue
    mid = msg.get('id')
    if mid and mid in seen:
        continue
    if mid:
        seen.add(mid)
    u = msg.get('usage', {}) or {}
    deduped['input']        += u.get('input_tokens', 0) or 0
    deduped['output']       += u.get('output_tokens', 0) or 0
    deduped['cache_create'] += u.get('cache_creation_input_tokens', 0) or 0
    deduped['cache_read']   += u.get('cache_read_input_tokens', 0) or 0

print(f'{"field":14} {"naive":>15} {"deduped":>15} {"reduction":>11}')
print('-' * 60)
for k in naive:
    factor = naive[k] / deduped[k] if deduped[k] else 1
    print(f'{k:14} {naive[k]:>15,} {deduped[k]:>15,} {factor:>10.2f}\u00d7')

## Step 4 — Now compare to `/usage`

Open Claude Code, run `/usage`, find your session, and **type the numbers below**.

> **⚠ Use a fresh `/usage` snapshot.** The JSONL grows every turn; if you paste numbers from an old screenshot they will undercount badly relative to the current file. Run `/usage` and re-run this notebook within a minute.

Format reminder (from the `/usage` output):
```
claude-opus-4-7:  2.2k input, 53.8k output, 11.3m cache read, 177.1k cache write ($8.10)
```

In [ ]:
USAGE = {
    'input':        2_200,        # the 'input' number from /usage
    'output':       53_800,
    'cache_create': 177_100,      # /usage calls this 'cache write'
    'cache_read':   11_300_000,
    'total_cost':   8.10,         # in USD
    'model':        'claude-opus-4-7',
}

def fmt_pct(a, b):
    if b == 0: return 'n/a'
    return f'{(a - b) / b * 100:+6.1f}%'

print(f'{"field":14} {"/usage":>14} {"deduped":>14} {"diff":>9}  within \u00b15%?')
print('-' * 65)
for k in ('input', 'output', 'cache_create', 'cache_read'):
    diff = fmt_pct(deduped[k], USAGE[k])
    ok = '\u2713' if abs(deduped[k] - USAGE[k]) <= 0.05 * max(USAGE[k], 1) else '\u2717'
    print(f'{k:14} {USAGE[k]:>14,} {deduped[k]:>14,} {diff:>9}    {ok}')

If your session looks anything like ours, **cache_read matches within ~1%** but **input** is way off (often the JSONL is missing thousands of input tokens that `/usage` reports). Let's find out why.

## Step 5 — Hunt the missing input tokens

The JSONL sums to a tiny `input` number (a few hundred) because of how prompt caching works: most of each turn's input is served from cache and reported under `cache_read_input_tokens`. The actual *uncached* input tokens are small.

But `/usage` reports a much larger number. Where do those extra tokens come from?

Hypothesis: **API calls that Claude Code makes but never persists to the JSONL.** Specifically, every time it generates a session title or agent name, it makes a separate API call. The *result* of that call appears as an `ai-title` or `agent-name` entry, but the per-call `usage` is only in the daemon's in-memory counter.

Let's count those ancillary calls and estimate their token contribution.

In [ ]:
n_ai_title = type_counts.get('ai-title', 0)
n_agent_name = type_counts.get('agent-name', 0)
n_aux = n_ai_title + n_agent_name

missing_input = USAGE['input'] - deduped['input']

print(f'Ancillary entries in JSONL: {n_aux} ({n_ai_title} ai-title + {n_agent_name} agent-name)')
print(f'JSONL input_tokens (deduped): {deduped["input"]:,}')
print(f'/usage input_tokens:          {USAGE["input"]:,}')
print(f'Difference:                   {missing_input:+,}')
print()

if missing_input > 0 and n_aux > 0:
    tokens_per_aux = missing_input / n_aux
    print(f'→ /usage shows {missing_input:,} more input tokens than the JSONL.')
    print(f'  If each of the {n_aux} ancillary calls used ~{tokens_per_aux:.0f} input tokens, that')
    print(f'  exactly closes the gap. Title/agent-name prompts are typically 50-100')
    print(f'  tokens, so a value in that range supports the hypothesis.')
elif missing_input < 0:
    print(f'→ The JSONL has MORE input tokens than your /usage snapshot.')
    print(f'  This means your /usage value is stale relative to the current JSONL.')
    print(f'  Re-run /usage in Claude Code right now and paste fresh numbers above,')
    print(f'  then re-execute this cell.')
else:
    print(f'→ No gap to explain.')

If the implied per-call cost is in the 30-100 token range, the hypothesis is consistent: the missing input tokens are from ancillary API calls Claude Code makes for session management.

**What this means:** no amount of clever parsing of the JSONL can recover these tokens, because **they aren't in the file**. They live only in the daemon's RAM counter.

## Step 6 — The smoking gun: Claude Code's daemon binary

Claude Code's binary contains string constants revealing how it tracks usage internally. Run this if your `claude` binary is at the standard location:

In [ ]:
import subprocess
import re
from pathlib import Path

# Find the claude binary
candidates = [
    HOME / '.local' / 'share' / 'claude' / 'versions',
    Path('/usr/local/bin/claude'),
    Path('/opt/homebrew/bin/claude'),
]

claude_bin = None
for c in candidates:
    if c.is_dir():
        versions = sorted([v for v in c.iterdir() if v.is_file()], reverse=True)
        if versions:
            claude_bin = versions[0]
            break
    elif c.is_file() or c.is_symlink():
        claude_bin = c.resolve()
        break

SYMBOLS = [
    'lastTotalInputTokens', 'lastTotalOutputTokens',
    'lastTotalCacheCreationInputTokens', 'lastTotalCacheReadInputTokens',
    'lastCost', 'lastModelUsage', 'lastSessionId', 'lastAPIDuration',
    'lastAPIDurationWithoutRetries',
]

if claude_bin and claude_bin.exists():
    print(f'Probing: {claude_bin}\n')
    result = subprocess.run(
        ['strings', str(claude_bin)],
        capture_output=True, text=True, timeout=30,
    )
    # Whole-word match against a short identifier-only line.
    # Skips minified-JS chunks that happen to contain these substrings.
    found = set()
    pattern = re.compile(r'^[A-Za-z0-9_]+$')
    for s in result.stdout.split('\n'):
        if 4 < len(s) < 60 and pattern.match(s) and s in SYMBOLS:
            found.add(s)
    print('Daemon-tracked counter symbols found in the binary:\n')
    for s in sorted(found):
        print(f'  {s}')
    print(f'\n({len(found)}/{len(SYMBOLS)} known symbols present)')
else:
    print('claude binary not found at standard paths; skipping')
    print('You can run manually: strings /path/to/claude | grep -wE "lastTotalInputTokens|lastCost"')

These `lastTotal*` symbols are the **runtime counters** the daemon maintains. They get incremented from every Anthropic API response the daemon receives — main conversation, title generation, agent-name generation, retries, anything. Only the *main conversation* responses get logged to JSONL.

Confirmed: `/usage` reads daemon RAM. The JSONL only has a subset of that data.

## Step 7 — The remaining gaps (output and cache_write)

Even after dedup, `output_tokens` typically sums higher than `/usage` reports, and `cache_create` is also higher. Let's check what's going on with each.

### Output: thinking tokens are in `output_tokens` but redacted from content

Claude 4 supports extended thinking. Thinking-mode tokens are counted in `output_tokens` but the actual thinking *content* gets redacted in the JSONL (you'll see `"thinking": ""` even when the count says hundreds of tokens were generated).

In [ ]:
thinking_chars = text_chars = tool_use_chars = 0
seen_for_content = set()
for e in ENTRIES:
    msg = e.get('message', {}) or {}
    if msg.get('role') != 'assistant':
        continue
    for c in msg.get('content') or []:
        t = c.get('type')
        if t == 'thinking':
            thinking_chars += len(c.get('thinking', '') or '')
        elif t == 'text':
            text_chars += len(c.get('text', '') or '')
        elif t == 'tool_use':
            tool_use_chars += len(json.dumps(c.get('input', {}) or {}))

approx_tokens = lambda c: c // 4  # rough 4 chars per token
visible_output = approx_tokens(text_chars + tool_use_chars)
approx_thinking = approx_tokens(thinking_chars)

print(f'Output content actually in JSONL (rough token estimate):')
print(f'  thinking content:      {approx_thinking:>10,} tokens  ({thinking_chars:,} chars)')
print(f'  text content:          {approx_tokens(text_chars):>10,} tokens  ({text_chars:,} chars)')
print(f'  tool_use args:         {approx_tokens(tool_use_chars):>10,} tokens  ({tool_use_chars:,} chars)')
print(f'  visible total:         {visible_output:>10,} tokens')
print(f'\nReported output_tokens (deduped): {deduped["output"]:,}')
print(f'Unaccounted-for tokens:           {deduped["output"] - visible_output:,}')
print(f'\n\u2192 The gap is thinking content that was redacted from JSONL.')

### Cache Write: 1h vs 5m ephemeral buckets

Anthropic's prompt cache has two TTL buckets:
- **5-minute ephemeral** — billed at 1.25× input price
- **1-hour ephemeral** — billed at 2× input price

The JSONL reports both under `cache_creation_input_tokens` but also breaks them out in a `cache_creation` sub-object. Claude Code's `/usage` may normalize these to 5m-equivalents (or use only the 5m portion) — which is why our raw sum is higher.

Let's see how your cache writes are distributed:

In [ ]:
cc_5m = cc_1h = 0
seen = set()
for e in ENTRIES:
    msg = e.get('message', {}) or {}
    if msg.get('role') != 'assistant':
        continue
    mid = msg.get('id')
    if mid and mid in seen:
        continue
    if mid:
        seen.add(mid)
    u = msg.get('usage', {}) or {}
    cc = u.get('cache_creation', {}) or {}
    cc_5m += cc.get('ephemeral_5m_input_tokens', 0) or 0
    cc_1h += cc.get('ephemeral_1h_input_tokens', 0) or 0

print(f'Cache writes by TTL bucket:')
print(f'  5-minute ephemeral:  {cc_5m:>10,} tokens (1.25\u00d7 input price)')
print(f'  1-hour ephemeral:    {cc_1h:>10,} tokens (2\u00d7 input price)')
print(f'  Total:               {cc_5m + cc_1h:>10,} tokens')
print(f'\n/usage cache write:    {USAGE["cache_create"]:>10,} tokens')
ratio = USAGE['cache_create'] / (cc_5m + cc_1h) if (cc_5m + cc_1h) else 0
print(f'Ratio:                 {ratio:.3f}')

If your session is heavily 1h-ephemeral, you'll see a ratio well below 1.0 — `/usage` is reporting fewer tokens than your raw sum, consistent with normalization or bucket-specific accounting in the daemon.

## Step 8 — Cost reconciliation

Even though three of four token fields don't match, **total cost** usually does, because cache_read dominates and matches cleanly. Let's verify with current Opus 4.7 pricing:

In [ ]:
PRICING = {
    'claude-opus-4-7':  {'input': 5.00, 'output': 25.00, 'cache_write_5m': 6.25, 'cache_write_1h': 10.00, 'cache_read': 0.50},
    'claude-opus-4-6':  {'input': 5.00, 'output': 25.00, 'cache_write_5m': 6.25, 'cache_write_1h': 10.00, 'cache_read': 0.50},
    'claude-opus-4-5':  {'input': 5.00, 'output': 25.00, 'cache_write_5m': 6.25, 'cache_write_1h': 10.00, 'cache_read': 0.50},
    'claude-opus-4-1':  {'input': 15.00, 'output': 75.00, 'cache_write_5m': 18.75, 'cache_write_1h': 30.00, 'cache_read': 1.50},
    'claude-opus-4-0':  {'input': 15.00, 'output': 75.00, 'cache_write_5m': 18.75, 'cache_write_1h': 30.00, 'cache_read': 1.50},
    'claude-sonnet-4':  {'input': 3.00, 'output': 15.00, 'cache_write_5m': 3.75, 'cache_write_1h': 6.00, 'cache_read': 0.30},
    'claude-haiku-4-5': {'input': 1.00, 'output': 5.00, 'cache_write_5m': 1.25, 'cache_write_1h': 2.00, 'cache_read': 0.10},
}

def find_pricing(model_id):
    # Longest-prefix match against known model families
    for prefix in sorted(PRICING.keys(), key=len, reverse=True):
        if model_id.startswith(prefix):
            return PRICING[prefix]
    return PRICING['claude-sonnet-4']  # default

model = USAGE['model']
p = find_pricing(model)
perM = 1_000_000

cost_breakdown = {
    'input':       deduped['input']  * p['input'] / perM,
    'output':      deduped['output'] * p['output'] / perM,
    'cache_5m':    cc_5m * p['cache_write_5m'] / perM,
    'cache_1h':    cc_1h * p['cache_write_1h'] / perM,
    'cache_read':  deduped['cache_read'] * p['cache_read'] / perM,
}
total = sum(cost_breakdown.values())

print(f'Cost reconstruction for {model}:')
for k, v in cost_breakdown.items():
    print(f'  {k:14} ${v:>8.4f}')
print(f'  {"TOTAL":14} ${total:>8.4f}')
print(f'  {"/usage":14} ${USAGE["total_cost"]:>8.4f}')
delta_pct = (total - USAGE['total_cost']) / USAGE['total_cost'] * 100
print(f'  {"delta":14}  {delta_pct:+.1f}%')

## Summary

Three takeaways for anyone building a token-counting tool on Claude Code's JSONL:

### 1. Always dedup by `message.id`

Claude Code splits one assistant message into multiple JSONL entries — one per content block. All fragments carry the *same* `usage` object. Without dedup, your totals are 2-3× too high.

### 2. `/usage` is not derivable from the JSONL

`/usage` reads the daemon's in-memory counter, which includes API calls Claude Code makes but never persists to disk:
- Session title generation (one call per session)
- Agent name generation
- Retries (the daemon also tracks `lastAPIDurationWithoutRetries` separately, implying the main counter includes them)

If your tool reads only the JSONL, you will undercount input tokens by anywhere from 5× to 20× depending on how many ancillary calls happened.

### 3. Cache and thinking accounting is bucketed

`cache_creation_input_tokens` mixes 5-minute and 1-hour ephemeral buckets (priced at 1.25× and 2× input respectively). The `cache_creation.ephemeral_*_input_tokens` sub-fields break them out. `/usage` may normalize across buckets. `output_tokens` includes thinking tokens whose content gets redacted from the JSONL — you can see the counts but not the text.

### What does match

- **Cache Read** (the dominant cost driver) matches `/usage` within ~1%.
- **Total cost** typically matches within ~2%.

So if you only care about cost, the JSONL is fine. If you need the per-field breakdown to agree with `/usage`, you'll need to read daemon state directly (Unix socket protocol, undocumented) or intercept Anthropic API responses (mitmproxy).

---

**Source code:** [github.com/odedha-dr/claude-watch](https://github.com/odedha-dr/claude-watch) — the dashboard this investigation came out of. The three bugs uncovered here (Opus 4.7 pricing fallthrough, fragment over-counting, and the new Context Window metric) are all fixed in the latest commit.